In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

model.invoke([HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?")])

AIMessage(content="저는 실시간 주가 데이터를 가지고 있지 않기 때문에 현재 테슬라 주가가 한 달 전에 비해 올랐는지 내렸는지 정확하게 말씀드릴 수 없습니다.\n\n주가는 매일 변동하므로, 정확한 정보를 확인하시려면 아래와 같은 실시간 금융 정보를 제공하는 웹사이트나 앱을 이용하시는 것이 좋습니다:\n\n*   **네이버 금융, 카카오 증권, 다음 금융** (한국 서비스)\n*   **Google Finance, Yahoo Finance, Bloomberg, Reuters** (글로벌 서비스)\n*   사용하시는 증권사 앱\n\n이곳에서 '테슬라 주가' 또는 'TSLA'를 검색하시면 현재 주가와 과거 데이터를 비교하여 변동 추이를 확인하실 수 있습니다.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--1ae356ff-761b-4cad-bd99-68b38dc416dd-0', usage_metadata={'input_tokens': 17, 'output_tokens': 429, 'total_tokens': 446, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 275}})

In [4]:
from langchain_core.tools import tool
from datetime import datetime
from zoneinfo import ZoneInfo

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    target_timezone = ZoneInfo(timezone)
    now = datetime.now(target_timezone).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [5]:
# 도구의 스키마(schema) 정의, LLM이 정확한 도구 호출 생성 가능
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: TSLA)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [6]:
import yfinance as yf

# 테슬라 주식 데이터 가져오기
tsla = yf.Ticker("TSLA")

# 1년간의 일별 주가 데이터
hist = tsla.history(period="1y")

In [8]:
type(hist)

pandas.core.frame.DataFrame

In [9]:
hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-11-29 00:00:00-05:00,336.079987,345.450012,334.649994,345.160004,37167600,0.0,0.0
2024-12-02 00:00:00-05:00,352.380005,360.000000,351.149994,357.089996,77986500,0.0,0.0
2024-12-03 00:00:00-05:00,351.799988,355.690002,348.200012,351.420013,58267200,0.0,0.0
2024-12-04 00:00:00-05:00,353.000000,358.100006,348.600006,357.929993,50810900,0.0,0.0
2024-12-05 00:00:00-05:00,359.869995,375.429993,359.500000,369.489990,81403600,0.0,0.0
...,...,...,...,...,...,...,...
2025-11-21 00:00:00-05:00,402.320007,402.799988,383.760010,391.089996,100460600,0.0,0.0
2025-11-24 00:00:00-05:00,402.170013,421.720001,401.089996,417.779999,96806400,0.0,0.0
2025-11-25 00:00:00-05:00,414.420013,420.480011,405.950012,419.399994,71915600,0.0,0.0


In [11]:
hist.to_markdown()

'| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2024-11-29 00:00:00-05:00 | 336.08 | 345.45 | 334.65 |  345.16 | 3.71676e+07 |           0 |              0 |\n| 2024-12-02 00:00:00-05:00 | 352.38 | 360    | 351.15 |  357.09 | 7.79865e+07 |           0 |              0 |\n| 2024-12-03 00:00:00-05:00 | 351.8  | 355.69 | 348.2  |  351.42 | 5.82672e+07 |           0 |              0 |\n| 2024-12-04 00:00:00-05:00 | 353    | 358.1  | 348.6  |  357.93 | 5.08109e+07 |           0 |              0 |\n| 2024-12-05 00:00:00-05:00 | 359.87 | 375.43 | 359.5  |  369.49 | 8.14036e+07 |           0 |              0 |\n| 2024-12-06 00:00:00-05:00 | 377.42 | 389.49 | 370.8  |  389.22 | 8.14558e+07 |           0 |              0 |\n| 2024-12-09 00:00:00-05:00 | 397.61 | 404.8  | 378.01 |  389.79 | 9.63592e+07 |           0 | 

In [12]:
@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

In [14]:
from langchain.agents import create_agent

# 도구들을 tools 리스트에 추가
tools = [get_current_time, get_yf_stock_history]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [15]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"}]},
)

In [16]:
result

{'messages': [HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐나 내렸나?', additional_kwargs={}, response_metadata={}, id='fc0b570f-d6a4-4043-9ea1-f6618425a677'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"ticker": "TSLA", "period": "1mo"}}'}, '__gemini_function_call_thought_signatures__': {'189daf9c-8eaf-46ab-beba-bc043eaca629': 'CpsIAXLI2nw2/0QG/Ew391Wle8qJ8jXxwrkofl70b9jRPLftJP0BX6bq06zkMPQrnXmjqiHDe86HMbGajs7jVumOw354ijHuSJboLvjBUdDQkkKGBhpaHmo4tprtBf9xAbeup5EHElkQNrZQ4I94l0Wz6oIBBS7ShnA8RMu12xTy0bxsJizrp0HkkFyQTPuGw3OOsnCralFg4Nr/MZW88y2FMPKjPsinTRe1sGSzbyEOwyy1yGI/HEchLSpWrgvTqqiOOV/LD6sCg//WZNW7eU8Qd0GgNISayxlmvJK0um9cviHIm1/ePIxsA/1B13GgcHyyb/wYw4GMhlma0nc6FJKDHqXdliW9ghty40O0m7I8hrbglfEkATihwFSGacpfPB7wKW13v9yreMZ5fyclQzjY5y6vkyWRuo8ad3l5qrCi8qejKX2Ez2nZzm92ZQBiCUTDNjZ6Uh283LeHAt8TLf4sgBrwL2cC2ZNRlw7Cd6uAmN1Catq4VN0IxoJhmMzKqDJ6/J6uBDxcaDjTzOXnbJDINyJynBbxfqGUQH82R/DOHWILko769iaG9GN1XVuNHivz7LO7jPa

In [24]:
result['messages'][-1].content[0]["text"]

'테슬라의 주가는 한달 전에 비해 **내렸습니다.**\n\n한 달 전인 2025년 10월 29일 종가는 461.51이었으나, 가장 최근인 2025년 11월 28일 종가는 430.17입니다.'